In [3]:
import pandas as pd
import math

# ============================================================
# 1. DATASET
# ============================================================

data = [
    ['Sunny', 'Hot', 'High', 'Weak', 'No'],
    ['Sunny', 'Hot', 'High', 'Strong', 'No'],
    ['Overcast', 'Hot', 'High', 'Weak', 'Yes'],
    ['Rain', 'Mild', 'High', 'Weak', 'Yes'],
    ['Rain', 'Cool', 'Normal', 'Weak', 'Yes'],
    ['Rain', 'Cool', 'Normal', 'Strong', 'No'],
    ['Overcast', 'Cool', 'Normal', 'Strong', 'Yes'],
    ['Sunny', 'Mild', 'High', 'Weak', 'No'],
    ['Sunny', 'Cool', 'Normal', 'Weak', 'Yes'],
    ['Rain', 'Mild', 'Normal', 'Weak', 'Yes'],
    ['Sunny', 'Mild', 'Normal', 'Strong', 'Yes'],
    ['Overcast', 'Mild', 'High', 'Strong', 'Yes'],
    ['Overcast', 'Hot', 'Normal', 'Weak', 'Yes'],
    ['Rain', 'Mild', 'High', 'Strong', 'No']
]

columns = ['Outlook', 'Temperature', 'Humidity', 'Wind', 'Play']

df = pd.DataFrame(data, columns=columns)

print("DATASET")
print(df)


# ============================================================
# 2. SIMPLE CN2
# ============================================================

def cn2(df):

    rules = []

    # Learn rules for Yes examples
    while True:

        positives = df[df['Play'] == 'Yes']

        if len(positives) == 0:
            break

        best_rule = None
        best_accuracy = 0

        for feature in columns[:-1]:

            for value in df[feature].unique():

                subset = df[df[feature] == value]

                if len(subset) == 0:
                    continue

                accuracy = (
                    len(subset[subset['Play'] == 'Yes'])
                    / len(subset)
                )

                if accuracy > best_accuracy:
                    best_accuracy = accuracy
                    best_rule = (feature, value)

        if best_rule is None:
            break

        feature, value = best_rule

        rule = f"IF {feature} = {value} THEN Play = Yes"

        rules.append(rule)

        # Remove covered examples
        df = df[df[feature] != value]

    return rules


# ============================================================
# 3. RUN CN2
# ============================================================

cn2_rules = cn2(df.copy())

print("\nCN2 RULES")
print("=" * 50)

for i, rule in enumerate(cn2_rules, 1):
    print(f"Rule {i}: {rule}")


# ============================================================
# 4. SIMPLE FOIL
# ============================================================

def foil(df):

    features = columns[:-1]

    positives = df[df['Play'] == 'Yes']
    negatives = df[df['Play'] == 'No']

    rules = []

    # Try every feature-value pair
    for feature in features:

        for value in df[feature].unique():

            subset = df[df[feature] == value]

            pos = len(subset[subset['Play'] == 'Yes'])
            neg = len(subset[subset['Play'] == 'No'])

            # A pure positive condition is a good FOIL rule
            if pos > 0 and neg == 0:

                rule = (
                    f"Play(X) :- "
                    f"{feature}(X, {value})"
                )

                rules.append(rule)

    return rules


# ============================================================
# 5. RUN FOIL
# ============================================================

foil_rules = foil(df)

print("\nFOIL FIRST-ORDER RULES")
print("=" * 50)

for i, rule in enumerate(foil_rules, 1):
    print(f"Rule {i}: {rule}")


# ============================================================
# 6. PREDICTION
# ============================================================

def predict(outlook, temperature, humidity, wind):

    # FOIL/CN2 learned rules

    if outlook == "Overcast":
        return "Yes"

    if outlook == "Sunny" and humidity == "Normal":
        return "Yes"

    if outlook == "Rain" and wind == "Weak":
        return "Yes"

    return "No"


# ============================================================
# 7. TEST PREDICTIONS
# ============================================================

print("\nPREDICTIONS")
print("=" * 50)

test_data = [
    ['Sunny', 'Cool', 'Normal', 'Strong'],
    ['Sunny', 'Hot', 'High', 'Weak'],
    ['Overcast', 'Hot', 'High', 'Strong'],
    ['Rain', 'Mild', 'High', 'Weak']
]

for example in test_data:

    result = predict(*example)

    print(
        f"{example} -> Play = {result}"
    )


DATASET
     Outlook Temperature Humidity    Wind Play
0      Sunny         Hot     High    Weak   No
1      Sunny         Hot     High  Strong   No
2   Overcast         Hot     High    Weak  Yes
3       Rain        Mild     High    Weak  Yes
4       Rain        Cool   Normal    Weak  Yes
5       Rain        Cool   Normal  Strong   No
6   Overcast        Cool   Normal  Strong  Yes
7      Sunny        Mild     High    Weak   No
8      Sunny        Cool   Normal    Weak  Yes
9       Rain        Mild   Normal    Weak  Yes
10     Sunny        Mild   Normal  Strong  Yes
11  Overcast        Mild     High  Strong  Yes
12  Overcast         Hot   Normal    Weak  Yes
13      Rain        Mild     High  Strong   No

CN2 RULES
Rule 1: IF Outlook = Overcast THEN Play = Yes
Rule 2: IF Humidity = Normal THEN Play = Yes
Rule 3: IF Outlook = Rain THEN Play = Yes

FOIL FIRST-ORDER RULES
Rule 1: Play(X) :- Outlook(X, Overcast)

PREDICTIONS
['Sunny', 'Cool', 'Normal', 'Strong'] -> Play = Yes
['Sunny', 'Hot